# 12 — Official Forecast Export  
## Gold Nexus Alpha — JSON-First Forecast Page Builder

**Purpose:** Use Notebook 11’s selected model artifact to export the official forecast JSON and page JSON.

This notebook does **not** hardcode the winning model. It reads:
- `artifacts/validation/selected_model_summary.json`
- `artifacts/validation/model_ranking.json`
- the selected model’s forecast path artifact

**Outputs:**
- `artifacts/forecast/official_forecast.json`
- `artifacts/forecast/official_forecast_path.csv`
- `artifacts/pages/page_official_forecast.json`

Run this after Notebook 11 finishes successfully.

## CELL 1 — Repo sync and setup

In [ ]:
# =========================
# CELL 1 — REPO SYNC
# =========================

import os
import subprocess
from pathlib import Path
from datetime import datetime, timezone

try:
    from google.colab import userdata
    IN_COLAB = True
except Exception:
    IN_COLAB = False

# ---- PROJECT SETTINGS ----
# These are set to your current GitHub repo. You can still override them with environment variables if needed.
GITHUB_USERNAME = os.environ.get("GITHUB_USERNAME", "rathee000001")
REPO_NAME = os.environ.get("REPO_NAME", "nyit-gold-intelligence-2026")
BRANCH = os.environ.get("BRANCH", "main")

REPO_URL_PUBLIC = f"https://github.com/{GITHUB_USERNAME}/{REPO_NAME}.git"

if IN_COLAB:
    LOCAL_REPO_DIR = Path("/content") / REPO_NAME
else:
    # If you are already inside the repo in VS Code, use the current folder.
    LOCAL_REPO_DIR = Path.cwd() if (Path.cwd() / ".git").exists() else Path.cwd() / REPO_NAME

def run_cmd(cmd, cwd=None, check=True):
    print(f"$ {' '.join(cmd)}")
    result = subprocess.run(cmd, cwd=cwd, text=True, capture_output=True)
    if result.stdout:
        print(result.stdout[-4000:])
    if result.stderr:
        print(result.stderr[-4000:])
    if check and result.returncode != 0:
        raise RuntimeError(f"Command failed: {' '.join(cmd)}")
    return result

if IN_COLAB:
    GITHUB_TOKEN = userdata.get("GITHUB_TOKEN")
    if not GITHUB_TOKEN:
        raise ValueError("GITHUB_TOKEN not found in Colab Secrets. Add a secret named GITHUB_TOKEN before running.")
    REPO_URL_AUTH = f"https://{GITHUB_TOKEN}@github.com/{GITHUB_USERNAME}/{REPO_NAME}.git"
else:
    GITHUB_TOKEN = os.environ.get("GITHUB_TOKEN", "")
    REPO_URL_AUTH = REPO_URL_PUBLIC

if LOCAL_REPO_DIR.exists() and (LOCAL_REPO_DIR / ".git").exists():
    print(f"Repo exists: {LOCAL_REPO_DIR}")
    run_cmd(["git", "fetch", "origin"], cwd=LOCAL_REPO_DIR)
    run_cmd(["git", "checkout", BRANCH], cwd=LOCAL_REPO_DIR)
    # Rebase keeps local Colab commits cleaner than merge commits.
    pull_result = run_cmd(["git", "pull", "--rebase", "origin", BRANCH], cwd=LOCAL_REPO_DIR, check=False)
    if pull_result.returncode != 0:
        raise RuntimeError(
            "Git pull --rebase failed. Check for local uncommitted changes or conflicts, then rerun. "
            "In Colab this usually means the repo was edited during the session."
        )
else:
    if LOCAL_REPO_DIR.exists() and not (LOCAL_REPO_DIR / ".git").exists():
        print(f"Folder exists but is not a git repo: {LOCAL_REPO_DIR}")
        print("Using a clean clone folder instead.")
        LOCAL_REPO_DIR = LOCAL_REPO_DIR.with_name(LOCAL_REPO_DIR.name + "-clone")
    print(f"Cloning repo into: {LOCAL_REPO_DIR}")
    run_cmd(["git", "clone", "-b", BRANCH, REPO_URL_AUTH, str(LOCAL_REPO_DIR)])

for folder in ["artifacts/validation", "artifacts/pages", "artifacts/models", "artifacts/forecast", "data/aligned"]:
    (LOCAL_REPO_DIR / folder).mkdir(parents=True, exist_ok=True)

print("✅ Repo sync complete.")
print(f"Local repo: {LOCAL_REPO_DIR}")

## CELL 2 — Dependencies

In [ ]:
# =========================
# CELL 2 — DEPENDENCIES
# =========================

import json
import math
import re
import warnings
from pathlib import Path
from datetime import datetime, timezone, date

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

try:
    from IPython.display import display
except Exception:
    def display(x):
        print(x)

warnings.filterwarnings("ignore")

pd.set_option("display.max_columns", 200)
pd.set_option("display.width", 200)

print("✅ Dependencies loaded.")

## CELL 3 — Load selected model and detect forecast source

In [ ]:
# =========================
# CELL 3 — LOAD SELECTED MODEL + DETECT FORECAST SOURCE
# =========================

REPO = LOCAL_REPO_DIR
MODELS_DIR = REPO / "artifacts/models"
FORECAST_DIR = REPO / "artifacts/forecast"
PAGES_DIR = REPO / "artifacts/pages"
VALIDATION_DIR = REPO / "artifacts/validation"

def load_json(path, required=False):
    path = Path(path)
    if not path.exists():
        if required:
            raise FileNotFoundError(f"Required JSON not found: {path.relative_to(REPO) if str(path).startswith(str(REPO)) else path}")
        return None
    with open(path, "r", encoding="utf-8") as f:
        return json.load(f)

def write_json(path, obj):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    with open(path, "w", encoding="utf-8") as f:
        json.dump(clean_for_json(obj), f, indent=2, ensure_ascii=False)
    print(f"✅ Wrote {path.relative_to(REPO)}")

def first_existing_path(candidates, glob_patterns=None):
    for p in candidates:
        p = Path(p)
        if p.exists():
            return p
    glob_patterns = glob_patterns or []
    for pattern in glob_patterns:
        matches = sorted(MODELS_DIR.glob(pattern))
        if matches:
            return matches[0]
    return None

def clean_for_json(obj):
    if isinstance(obj, dict):
        return {str(k): clean_for_json(v) for k, v in obj.items()}
    if isinstance(obj, list):
        return [clean_for_json(v) for v in obj]
    if isinstance(obj, tuple):
        return [clean_for_json(v) for v in obj]
    if isinstance(obj, np.ndarray):
        return [clean_for_json(v) for v in obj.tolist()]
    if isinstance(obj, (pd.Timestamp, datetime, date)):
        return obj.isoformat()
    if obj is pd.NaT:
        return None
    if isinstance(obj, np.integer):
        return int(obj)
    if isinstance(obj, np.floating):
        if np.isnan(obj) or np.isinf(obj):
            return None
        return float(obj)
    if isinstance(obj, float):
        if math.isnan(obj) or math.isinf(obj):
            return None
        return obj
    try:
        if pd.isna(obj):
            return None
    except Exception:
        pass
    return obj

selected_model_summary = load_json(VALIDATION_DIR / "selected_model_summary.json", required=True)
model_ranking = load_json(VALIDATION_DIR / "model_ranking.json", required=True)
validation_summary = load_json(VALIDATION_DIR / "validation_summary.json", required=False) or {}

selected_model = selected_model_summary.get("selected_model", {})
selected_model_key = selected_model.get("model_key")

if not selected_model_key:
    raise ValueError("selected_model_summary.json does not contain selected_model.model_key. Re-run Notebook 11.")

OFFICIAL_CUTOFF_DATE = validation_summary.get("official_forecast_cutoff_date", "2026-03-31")
RUN_TIMESTAMP_UTC = datetime.now(timezone.utc).isoformat()

FORECAST_PATH_CANDIDATES_BY_MODEL = {
    "naive": [
        MODELS_DIR / "baseline_forecast_paths.json",
        MODELS_DIR / "naive_moving_average_forecast_paths.json",
        MODELS_DIR / "naive_forecast_path.json",
        MODELS_DIR / "baseline_forecast_path.json",
    ],
    "moving_average": [
        MODELS_DIR / "baseline_forecast_paths.json",
        MODELS_DIR / "naive_moving_average_forecast_paths.json",
        MODELS_DIR / "moving_average_forecast_path.json",
        MODELS_DIR / "baseline_forecast_path.json",
    ],
    "exponential_smoothing": [
        MODELS_DIR / "exponential_smoothing_forecast_path.json",
        MODELS_DIR / "ets_forecast_path.json",
    ],
    "regression": [
        MODELS_DIR / "regression_forecast_path.json",
        MODELS_DIR / "linear_regression_forecast_path.json",
    ],
    "arima": [
        MODELS_DIR / "arima_forecast_path.json",
    ],
    "sarimax": [
        MODELS_DIR / "sarimax_forecast_path.json",
    ],
    "xgboost": [
        MODELS_DIR / "xgboost_forecast_path.json",
        MODELS_DIR / "xgb_forecast_path.json",
    ],
    "prophet_optional": [
        MODELS_DIR / "prophet_forecast_path.json",
    ],
}

FORECAST_PATH_GLOBS_BY_MODEL = {
    "naive": ["*baseline*forecast*path*.json", "*naive*forecast*path*.json"],
    "moving_average": ["*baseline*forecast*path*.json", "*moving*average*forecast*path*.json"],
    "exponential_smoothing": ["*exponential*smoothing*forecast*path*.json", "*ets*forecast*path*.json"],
    "regression": ["*regression*forecast*path*.json"],
    "arima": ["*arima*forecast*path*.json"],
    "sarimax": ["*sarimax*forecast*path*.json"],
    "xgboost": ["*xgboost*forecast*path*.json", "*xgb*forecast*path*.json"],
    "prophet_optional": ["*prophet*forecast*path*.json"],
}

selected_forecast_path = first_existing_path(
    FORECAST_PATH_CANDIDATES_BY_MODEL.get(selected_model_key, []),
    FORECAST_PATH_GLOBS_BY_MODEL.get(selected_model_key, [])
)

print("Selected model from Notebook 11:")
display(pd.DataFrame([selected_model]))

print("Selected forecast source:")
display(pd.DataFrame([{
    "selected_model_key": selected_model_key,
    "forecast_path": str(selected_forecast_path.relative_to(REPO)) if selected_forecast_path else "not found",
    "official_cutoff_date": OFFICIAL_CUTOFF_DATE
}]))

## CELL 4 — Normalize forecast records

In [ ]:
# =========================
# CELL 4 — NORMALIZE FORECAST RECORDS
# =========================

def find_first_col(columns, terms, exact_first=True):
    cols = list(columns)

    if exact_first:
        normalized = {str(c).lower().strip(): c for c in cols}
        for term in terms:
            term_norm = str(term).lower().strip()
            if term_norm in normalized:
                return normalized[term_norm]

    for col in cols:
        c = str(col).lower()
        if any(str(t).lower() in c for t in terms):
            return col
    return None

def collect_record_lists(obj, path="root"):
    """
    Recursively collect list-of-dict blocks from a JSON artifact.
    """
    found = []

    if isinstance(obj, list):
        if all(isinstance(x, dict) for x in obj) and len(obj) > 0:
            found.append({"path": path, "records": obj, "count": len(obj)})
        return found

    if isinstance(obj, dict):
        for key, value in obj.items():
            found.extend(collect_record_lists(value, f"{path}.{key}"))

    return found

def score_record_list(item, model_key):
    records = item["records"]
    path = item["path"].lower()
    if not records:
        return -1

    sample_keys = set()
    for rec in records[:20]:
        sample_keys.update([str(k).lower() for k in rec.keys()])

    score = 0

    # Prefer paths that look model-specific or forecast-specific.
    model_terms = {
        "naive": ["naive"],
        "moving_average": ["moving", "average"],
        "exponential_smoothing": ["exponential", "smoothing", "ets"],
        "regression": ["regression"],
        "arima": ["arima"],
        "sarimax": ["sarimax"],
        "xgboost": ["xgboost", "xgb"],
        "prophet_optional": ["prophet"],
    }.get(model_key, [model_key])

    for term in model_terms:
        if term in path:
            score += 5

    if "forecast" in path:
        score += 4
    if "path" in path:
        score += 2
    if "record" in path or "data" in path:
        score += 1

    joined_keys = " ".join(sample_keys)
    if "date" in joined_keys or "ds" in joined_keys:
        score += 5
    if "forecast" in joined_keys or "prediction" in joined_keys or "yhat" in joined_keys:
        score += 5
    if "actual" in joined_keys or "gold_price" in joined_keys or "observed" in joined_keys:
        score += 2
    if "split" in joined_keys or "period" in joined_keys or "dataset" in joined_keys:
        score += 1

    # Larger record blocks are often actual chart/path data.
    score += min(item["count"] / 1000, 3)

    return score

def load_forecast_artifact(path):
    if path is None or not Path(path).exists():
        return None

    path = Path(path)
    if path.suffix.lower() == ".csv":
        df = pd.read_csv(path)
        return df, {"source_type": "csv", "record_block_path": "csv"}

    obj = load_json(path, required=True)
    candidates = collect_record_lists(obj)

    if not candidates:
        # Last fallback: try direct DataFrame conversion.
        try:
            df = pd.DataFrame(obj)
            if not df.empty:
                return df, {"source_type": "json_dataframe_fallback", "record_block_path": "root"}
        except Exception:
            pass
        return pd.DataFrame(), {"source_type": "json", "record_block_path": None}

    candidates = sorted(
        candidates,
        key=lambda item: score_record_list(item, selected_model_key),
        reverse=True
    )

    chosen = candidates[0]
    df = pd.DataFrame(chosen["records"])
    return df, {
        "source_type": "json",
        "record_block_path": chosen["path"],
        "candidate_record_blocks": [
            {"path": c["path"], "count": c["count"], "score": score_record_list(c, selected_model_key)}
            for c in candidates[:10]
        ]
    }

def filter_for_selected_model(df, model_key):
    if df is None or df.empty:
        return pd.DataFrame()

    model_col = find_first_col(df.columns, ["model_key", "model", "method", "candidate", "forecast_type"])
    if model_col is None:
        return df.copy()

    model_terms = {
        "naive": ["naive"],
        "moving_average": ["moving", "average"],
        "exponential_smoothing": ["exponential", "smoothing", "ets"],
        "regression": ["regression"],
        "arima": ["arima"],
        "sarimax": ["sarimax"],
        "xgboost": ["xgboost", "xgb"],
        "prophet_optional": ["prophet"],
    }.get(model_key, [model_key])

    text = df[model_col].astype(str).str.lower()
    mask = pd.Series(False, index=df.index)
    for term in model_terms:
        mask = mask | text.str.contains(term, na=False)

    filtered = df[mask].copy()

    # For moving average, require both words if possible to avoid picking naive rows.
    if model_key == "moving_average":
        strict_mask = text.str.contains("moving", na=False) & text.str.contains("average", na=False)
        strict_filtered = df[strict_mask].copy()
        if not strict_filtered.empty:
            filtered = strict_filtered

    return filtered if not filtered.empty else df.copy()

raw_forecast_df, forecast_source_info = load_forecast_artifact(selected_forecast_path)

if raw_forecast_df is None:
    raw_forecast_df = pd.DataFrame()

selected_forecast_df = filter_for_selected_model(raw_forecast_df, selected_model_key)

print("Raw forecast rows:", len(raw_forecast_df))
print("Selected-model forecast rows:", len(selected_forecast_df))
print("Forecast source info:")
print(json.dumps(clean_for_json(forecast_source_info), indent=2)[:2500])

display(selected_forecast_df.head(10) if not selected_forecast_df.empty else pd.DataFrame(columns=["status"]))

## CELL 5 — Standardize official forecast table

In [ ]:
# =========================
# CELL 5 — STANDARDIZE OFFICIAL FORECAST TABLE
# =========================

def standardize_forecast_df(df):
    if df is None or df.empty:
        return pd.DataFrame(), {
            "status": "empty_forecast_path",
            "warning": "No forecast path records were found for the selected model."
        }

    temp = df.copy()

    date_col = find_first_col(temp.columns, ["date", "ds"])
    actual_col = find_first_col(temp.columns, ["actual", "actual_gold_price", "y_true", "observed", "gold_price", "y"])
    forecast_col = find_first_col(temp.columns, ["forecast", "prediction", "predicted", "yhat", "y_pred", "forecast_gold_price"])
    lower_col = find_first_col(temp.columns, ["lower", "lower_bound", "yhat_lower", "forecast_lower", "lo"])
    upper_col = find_first_col(temp.columns, ["upper", "upper_bound", "yhat_upper", "forecast_upper", "hi"])
    split_col = find_first_col(temp.columns, ["split", "period", "dataset", "segment"])
    model_col = find_first_col(temp.columns, ["model_key", "model", "method", "candidate", "forecast_type"])

    notes = {
        "date_column": str(date_col) if date_col is not None else None,
        "actual_column": str(actual_col) if actual_col is not None else None,
        "forecast_column": str(forecast_col) if forecast_col is not None else None,
        "lower_column": str(lower_col) if lower_col is not None else None,
        "upper_column": str(upper_col) if upper_col is not None else None,
        "split_column": str(split_col) if split_col is not None else None,
        "model_column": str(model_col) if model_col is not None else None,
    }

    if date_col is None or forecast_col is None:
        notes["status"] = "missing_required_columns"
        notes["warning"] = "Forecast path exists, but Notebook 12 could not identify date and forecast columns."
        return pd.DataFrame(), notes

    out = pd.DataFrame()
    out["date"] = pd.to_datetime(temp[date_col], errors="coerce")
    out["official_forecast"] = pd.to_numeric(temp[forecast_col], errors="coerce")

    if actual_col is not None:
        out["actual_gold_price"] = pd.to_numeric(temp[actual_col], errors="coerce")
    else:
        out["actual_gold_price"] = np.nan

    if lower_col is not None:
        out["forecast_lower"] = pd.to_numeric(temp[lower_col], errors="coerce")
    else:
        out["forecast_lower"] = np.nan

    if upper_col is not None:
        out["forecast_upper"] = pd.to_numeric(temp[upper_col], errors="coerce")
    else:
        out["forecast_upper"] = np.nan

    if split_col is not None:
        out["split"] = temp[split_col].astype(str)
    else:
        out["split"] = np.where(out["date"] > pd.to_datetime(OFFICIAL_CUTOFF_DATE), "future_forecast", "historical_or_test")

    if model_col is not None:
        out["source_model_label"] = temp[model_col].astype(str)
    else:
        out["source_model_label"] = selected_model.get("model_name", selected_model_key)

    out["selected_model_key"] = selected_model_key
    out["selected_model_name"] = selected_model.get("model_name", selected_model_key)

    out = out.dropna(subset=["date", "official_forecast"]).sort_values("date").reset_index(drop=True)

    notes["status"] = "ok" if not out.empty else "no_valid_rows_after_standardization"
    notes["records_after_standardization"] = int(len(out))

    return out, notes

official_forecast_df, standardization_notes = standardize_forecast_df(selected_forecast_df)

cutoff_dt = pd.to_datetime(OFFICIAL_CUTOFF_DATE)
if not official_forecast_df.empty:
    future_mask = official_forecast_df["date"] > cutoff_dt
    future_forecast_df = official_forecast_df[future_mask].copy()
    historical_forecast_df = official_forecast_df[~future_mask].copy()
else:
    future_forecast_df = pd.DataFrame()
    historical_forecast_df = pd.DataFrame()

print("Standardization notes:")
print(json.dumps(clean_for_json(standardization_notes), indent=2))

print("Official forecast record counts:")
display(pd.DataFrame([{
    "total_records": int(len(official_forecast_df)),
    "records_on_or_before_cutoff": int(len(historical_forecast_df)),
    "records_after_cutoff": int(len(future_forecast_df)),
    "official_cutoff_date": OFFICIAL_CUTOFF_DATE,
}]))

display(official_forecast_df.head(10) if not official_forecast_df.empty else pd.DataFrame(columns=["status"]))

## CELL 6 — Export official forecast artifacts

In [ ]:
# =========================
# CELL 6 — EXPORT OFFICIAL FORECAST ARTIFACTS
# =========================

official_records = official_forecast_df.copy()
if not official_records.empty:
    official_records["date"] = official_records["date"].dt.strftime("%Y-%m-%d")

future_records = future_forecast_df.copy()
if not future_records.empty:
    future_records["date"] = future_records["date"].dt.strftime("%Y-%m-%d")

latest_record = None
next_after_cutoff_record = None

if not official_records.empty:
    latest_record = official_records.iloc[-1].to_dict()

if not future_records.empty:
    next_after_cutoff_record = future_records.iloc[0].to_dict()

export_status = "ok"
warnings_list = []

if selected_forecast_path is None:
    export_status = "missing_forecast_path"
    warnings_list.append("Selected model forecast path artifact was not found. Re-run the selected model notebook and push its forecast path JSON.")
elif official_forecast_df.empty:
    export_status = "forecast_path_unparsed"
    warnings_list.append("A forecast path artifact was found, but Notebook 12 could not standardize usable date/forecast records.")
elif future_forecast_df.empty:
    warnings_list.append(
        "No records after the official cutoff date were detected. The artifact still exports the selected model path for frontend display."
    )

official_forecast_artifact = {
    "artifact_type": "official_forecast_export",
    "generated_at_utc": RUN_TIMESTAMP_UTC,
    "export_status": export_status,
    "official_forecast_cutoff_date": OFFICIAL_CUTOFF_DATE,
    "selected_model": selected_model,
    "selection_source": {
        "selected_model_summary": "artifacts/validation/selected_model_summary.json",
        "model_ranking": "artifacts/validation/model_ranking.json",
    },
    "forecast_source": {
        "path": str(selected_forecast_path.relative_to(REPO)) if selected_forecast_path else None,
        "source_info": forecast_source_info,
        "standardization_notes": standardization_notes,
    },
    "record_counts": {
        "total_records": int(len(official_forecast_df)),
        "records_on_or_before_cutoff": int(len(historical_forecast_df)),
        "records_after_cutoff": int(len(future_forecast_df)),
    },
    "latest_record": latest_record,
    "next_forecast_after_cutoff": next_after_cutoff_record,
    "records": official_records.to_dict(orient="records") if not official_records.empty else [],
    "future_records_after_cutoff": future_records.to_dict(orient="records") if not future_records.empty else [],
    "warnings": warnings_list,
    "professor_safe_interpretation": [
        "The official forecast export uses the selected model from Notebook 11.",
        "The selected model is not hardcoded in the frontend.",
        "Frontend pages should read this JSON artifact directly.",
        "If future records are absent, the website should say the forecast path exists but no post-cutoff forecast was exported."
    ],
}

page_official_forecast = {
    "artifact_type": "page_official_forecast",
    "generated_at_utc": RUN_TIMESTAMP_UTC,
    "page_title": "Official Gold Forecast",
    "page_subtitle": "Selected-model forecast exported from the validation pipeline.",
    "export_status": export_status,
    "official_forecast_cutoff_date": OFFICIAL_CUTOFF_DATE,
    "selected_model": selected_model,
    "kpi_cards": [
        {
            "label": "Selected Model",
            "value": selected_model.get("model_name", selected_model_key),
            "description": "Chosen by Notebook 11 model comparison."
        },
        {
            "label": "Primary RMSE",
            "value": selected_model.get("primary_rmse"),
            "description": selected_model.get("ranking_basis", "Ranking basis from selected model summary.")
        },
        {
            "label": "Forecast Records",
            "value": int(len(official_forecast_df)),
            "description": "Standardized records exported for frontend charting."
        },
        {
            "label": "Future Records",
            "value": int(len(future_forecast_df)),
            "description": "Records after the official cutoff date."
        },
    ],
    "chart_data": official_records.to_dict(orient="records") if not official_records.empty else [],
    "future_chart_data": future_records.to_dict(orient="records") if not future_records.empty else [],
    "latest_record": latest_record,
    "next_forecast_after_cutoff": next_after_cutoff_record,
    "warnings": warnings_list,
    "json_sources": [
        "artifacts/forecast/official_forecast.json",
        "artifacts/validation/selected_model_summary.json",
        "artifacts/validation/model_ranking.json",
    ],
    "frontend_guidance": [
        "Use chart_data for the full selected model path.",
        "Use future_chart_data for post-cutoff forecast display if available.",
        "Do not hardcode selected model name, metrics, or forecast values in React components."
    ]
}

write_json(FORECAST_DIR / "official_forecast.json", official_forecast_artifact)
write_json(PAGES_DIR / "page_official_forecast.json", page_official_forecast)

csv_path = FORECAST_DIR / "official_forecast_path.csv"
if not official_records.empty:
    official_records.to_csv(csv_path, index=False)
else:
    pd.DataFrame(columns=[
        "date", "official_forecast", "actual_gold_price", "forecast_lower", "forecast_upper",
        "split", "source_model_label", "selected_model_key", "selected_model_name"
    ]).to_csv(csv_path, index=False)

print(f"✅ Wrote {csv_path.relative_to(REPO)}")
print("\nExport status:", export_status)
if warnings_list:
    print("Warnings:")
    for w in warnings_list:
        print("-", w)

## CELL 7 — GitHub push

In [ ]:
# =========================
# CELL 7 — GITHUB PUSH
# =========================

commit_message = "Add official forecast export artifacts"

run_cmd(["git", "config", "user.email", "colab@gold-nexus-alpha.local"], cwd=REPO)
run_cmd(["git", "config", "user.name", "Gold Nexus Alpha Colab"], cwd=REPO)

paths_to_add = [
    "artifacts/forecast/official_forecast.json",
    "artifacts/forecast/official_forecast_path.csv",
    "artifacts/pages/page_official_forecast.json",
]

for p in paths_to_add:
    if (REPO / p).exists():
        run_cmd(["git", "add", p], cwd=REPO, check=False)
    else:
        print(f"Skipping missing file: {p}")

status = run_cmd(["git", "status", "--short"], cwd=REPO, check=False)

if not status.stdout.strip():
    print("No changes to commit.")
else:
    commit_result = run_cmd(["git", "commit", "-m", commit_message], cwd=REPO, check=False)

    if commit_result.returncode != 0:
        print("No commit created, possibly because files are unchanged.")
    else:
        pull_result = run_cmd(["git", "pull", "--rebase", "origin", BRANCH], cwd=REPO, check=False)
        if pull_result.returncode != 0:
            raise RuntimeError(
                "Git pull --rebase failed before push. Resolve conflicts, then run: git rebase --continue and git push origin main."
            )

        run_cmd(["git", "push", "origin", BRANCH], cwd=REPO)
        print("✅ Official forecast artifacts pushed to GitHub.")

# End of Notebook 12

After this notebook finishes:
- the official forecast artifacts are under `artifacts/forecast`
- the frontend-ready page JSON is `artifacts/pages/page_official_forecast.json`

Next frontend step: create the Official Forecast / Validation page components that read these JSON files only.